# AI-4002 Computer Vision Lab — Open-Book Exam Master Notebook

**Exam use:** MCQs + 1 theory question + coding questions.

Ye notebook intentionally **template/cheat-sheet style** hai: har common task ka ready pattern diya gaya hai, jise exam mein filename, parameters, thresholds, points, etc. change karke adapt kar sakte ho.

**Covered from uploaded material:** Lab 01, Lab 03 (Geometric Transformations), Lab 04 (Feature Extraction / Filtering / Convolution / Edge Detection), and the later section labelled Lab 05 (Wavelets, Boundary Detection, Hough Transform, SIFT).  
**Lab 02:** manual unavailable; exam note says Photometric/Point Transformations: **Log, Gamma, Piecewise, Histogram**. These are included separately.

> Before exam: keep the notebook and image files in the same folder, or change `IMAGE_PATH`.

## 0. Super-short mental map

- **Photometric / point transformation:** pixel ki value change, location same.
- **Geometric transformation:** pixel ki location change.
- **Feature extraction:** image se useful compact information nikalna.
- **Segmentation:** image ko meaningful regions mein divide karna.
- **Edge detection:** intensity/color ke rapid changes → boundaries.
- **Hough:** lines/circles jaisi shapes detect.
- **SIFT:** scale/rotation-robust local features + descriptors.

In [ ]:
# MASTER IMPORTS
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Optional packages used in later labs
# If needed in your environment:
# !pip install scikit-image PyWavelets

IMAGE_PATH = "image.jpg"
IMAGE2_PATH = "image2.jpg"

image = cv2.imread(IMAGE_PATH)
if image is None:
    print("Image not found. Change IMAGE_PATH.")
else:
    print("Shape:", image.shape)

# LAB 01 — Basics + Digital Image Processing

### Must-know
`imread`, `cvtColor`, `shape`, `resize`, `GaussianBlur`, NumPy slicing/cropping, `putText`, `reshape`, Pandas statistics, `threshold`, rotation, `add`, histogram equalization.

**OpenCV color order:** BGR.  
**Matplotlib display:** usually RGB.

In [ ]:
# Read + display image
image = cv2.imread(IMAGE_PATH)

if image is None:
    raise FileNotFoundError(IMAGE_PATH)

image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(7,5))
plt.imshow(image_rgb)
plt.title("Original Image")
plt.axis("off")
plt.show()

In [ ]:
# Grayscale
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

plt.imshow(gray, cmap="gray")
plt.title("Grayscale")
plt.axis("off")
plt.show()

print("Color shape:", image.shape)
print("Gray shape:", gray.shape)

In [ ]:
# Resize — easiest exam template
scale = 0.5
resized = cv2.resize(image, None, fx=scale, fy=scale)

print("Original:", image.shape[:2], "(height, width)")
print("Resized :", resized.shape[:2])

plt.imshow(cv2.cvtColor(resized, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

### `shape` reminder

For a color image:

`image.shape = (height, width, channels)`

For grayscale:

`gray.shape = (height, width)`

Exam trap: Python/OpenCV array order is **height, width**, not width, height.

In [ ]:
# Gaussian Blur
# Kernel sizes should normally be odd: 3x3, 5x5, 31x31, ...
blur = cv2.GaussianBlur(image, (31, 31), 0)

plt.imshow(cv2.cvtColor(blur, cv2.COLOR_BGR2RGB))
plt.title("Gaussian Blur")
plt.axis("off")
plt.show()

In [ ]:
# Crop — NumPy slicing
rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
h, w = rgb.shape[:2]

# General syntax:
# crop = image[y1:y2, x1:x2]

# Left half
left_half = rgb[:, :w//2]

# Right half
right_half = rgb[:, w//2:]

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.imshow(left_half); plt.title("Left Half"); plt.axis("off")
plt.subplot(1,2,2)
plt.imshow(right_half); plt.title("Right Half"); plt.axis("off")
plt.show()

In [ ]:
# Add text
annotated = rgb.copy()

cv2.putText(
    annotated,
    "Computer Vision",
    (50, 80),                         # (x, y)
    cv2.FONT_HERSHEY_SIMPLEX,
    1.5,                              # font scale
    (255, 0, 0),                      # RGB because annotated is RGB
    2                                 # thickness
)

plt.imshow(annotated)
plt.axis("off")
plt.show()

In [ ]:
# Analyze pixels with Pandas
flat = image.reshape(-1, 3)  # each row = one pixel; columns = B,G,R
df = pd.DataFrame(flat, columns=["B", "G", "R"])

display(df.describe())

In [ ]:
# Binary threshold
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

ret, binary = cv2.threshold(
    gray,
    100,                  # threshold
    200,                  # maximum value
    cv2.THRESH_BINARY
)

print("Threshold used:", ret)

plt.imshow(binary, cmap="gray")
plt.title("Binary Threshold")
plt.axis("off")
plt.show()

### Threshold rule

For `cv2.THRESH_BINARY`:

- pixel > threshold → `maxval`
- otherwise → `0`

So with threshold=100 and maxval=200:

`80 -> 0`, `100 -> 0`, `150 -> 200`.

Common exam change: they may ask for threshold 128, 150, 200 etc.

In [ ]:
# Rotation
rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
h, w = rgb.shape[:2]

center = (w/2, h/2)
angle = 45
scale = 1

M = cv2.getRotationMatrix2D(center, angle, scale)
rotated = cv2.warpAffine(rgb, M, (w, h))

plt.imshow(rotated)
plt.title("Rotated")
plt.axis("off")
plt.show()

In [ ]:
# Image blending using cv2.add()
img1 = cv2.imread(IMAGE_PATH)
img2 = cv2.imread(IMAGE2_PATH)

if img1 is None or img2 is None:
    print("Change IMAGE_PATH / IMAGE2_PATH.")
else:
    img1 = cv2.resize(img1, (img2.shape[1], img2.shape[0]))
    blended = cv2.add(img1, img2)

    plt.imshow(cv2.cvtColor(blended, cv2.COLOR_BGR2RGB))
    plt.title("Blended")
    plt.axis("off")
    plt.show()

In [ ]:
# Histogram equalization
gray = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)
equalized = cv2.equalizeHist(gray)

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.imshow(gray, cmap="gray"); plt.title("Original Gray"); plt.axis("off")
plt.subplot(1,2,2)
plt.imshow(equalized, cmap="gray"); plt.title("Equalized"); plt.axis("off")
plt.show()

# LAB 02 — Photometric / Point Transformations

**Important exam note supplied by you:** Lab 02 manual is unavailable, but these are included:
1. Log
2. Gamma
3. Piecewise
4. Histogram

### Core idea
Photometric transformation changes **pixel intensity/value**, not the pixel's spatial location.

General idea:

`g(x,y) = T(f(x,y))`

#### Log
`g = c * log(1 + r)`

Usually expands dark intensity differences.

#### Gamma / power law
`g = c * r^gamma`

After normalization to [0,1]:
- gamma < 1 → brighter
- gamma > 1 → darker
- gamma = 1 → approximately unchanged

#### Piecewise
Different linear equations are used over different intensity ranges. A common use is contrast stretching.

#### Histogram
Histogram = distribution/count of intensity values. Histogram equalization redistributes intensities to improve global contrast.

In [ ]:
# LOG TRANSFORMATION — manual NumPy template
gray = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)

r = gray.astype(np.float32) / 255.0
c = 1.0

log_img = c * np.log1p(r)
log_img = cv2.normalize(log_img, None, 0, 255, cv2.NORM_MINMAX)
log_img = log_img.astype(np.uint8)

plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.imshow(gray, cmap="gray"); plt.title("Original"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(log_img, cmap="gray"); plt.title("Log Transform"); plt.axis("off")
plt.show()

In [ ]:
# GAMMA TRANSFORMATION — exam-ready NumPy template
gray = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)

gamma = 0.5  # change to 2.0 etc.
r = gray.astype(np.float32) / 255.0

gamma_img = np.power(r, gamma)
gamma_img = (gamma_img * 255).clip(0, 255).astype(np.uint8)

plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.imshow(gray, cmap="gray"); plt.title("Original"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(gamma_img, cmap="gray"); plt.title(f"Gamma={gamma}"); plt.axis("off")
plt.show()

In [ ]:
# GAMMA — skimage shortcut
# !pip install scikit-image
from skimage import exposure, img_as_float

gray = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)
gamma = 0.5

gamma_img = exposure.adjust_gamma(gray, gamma)

plt.imshow(gamma_img, cmap="gray")
plt.title(f"Gamma={gamma}")
plt.axis("off")
plt.show()

In [ ]:
# LOG — skimage shortcut
from skimage import exposure

gray = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)
log_img = exposure.adjust_log(gray, gain=1)

plt.imshow(log_img, cmap="gray")
plt.title("Log Correction")
plt.axis("off")
plt.show()

In [ ]:
# PIECEWISE LINEAR CONTRAST STRETCHING
# General 3-region template.
# r1,r2 = input breakpoints; s1,s2 = output breakpoints.

gray = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE).astype(np.float32)

r1, s1 = 70, 0
r2, s2 = 180, 255

out = np.zeros_like(gray)

# region 1
mask1 = gray <= r1
out[mask1] = (s1 / r1) * gray[mask1] if r1 != 0 else s1

# region 2
mask2 = (gray > r1) & (gray <= r2)
out[mask2] = ((s2-s1)/(r2-r1)) * (gray[mask2]-r1) + s1

# region 3
mask3 = gray > r2
out[mask3] = ((255-s2)/(255-r2)) * (gray[mask3]-r2) + s2

out = np.clip(out, 0, 255).astype(np.uint8)

plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.imshow(gray, cmap="gray"); plt.title("Original"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(out, cmap="gray"); plt.title("Piecewise Contrast Stretch"); plt.axis("off")
plt.show()

In [ ]:
# HISTOGRAM — basic plot
gray = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)

plt.hist(gray.ravel(), bins=256, range=(0,256))
plt.xlabel("Intensity")
plt.ylabel("Pixel count")
plt.title("Histogram")
plt.show()

# LAB 03 — Geometric Transformations

Manual scope:
- image coordinate system
- 2D vs 3D
- transformation matrix
- affine transformations
- linear transformations
- scaling, rotation, reflection, shearing
- translation
- homogeneous coordinates
- projective transformation / homography
- perspective

**Most important distinction:**

Photometric = *what the pixel looks like*  
Geometric = *where the pixel is located*

Image coordinate system in the manual:
- origin `(0,0)` = top-left
- x increases → right
- y increases → down
- array access is `image[y, x]`

In [ ]:
# AFFINE: ROTATION
image = cv2.imread(IMAGE_PATH)
h, w = image.shape[:2]

center = (w/2, h/2)
M = cv2.getRotationMatrix2D(center, 45, 1.0)
rotated = cv2.warpAffine(image, M, (w, h))

plt.imshow(cv2.cvtColor(rotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
# AFFINE: SCALING
image = cv2.imread(IMAGE_PATH)

scale_x = 3
scale_y = 3

scaled = cv2.resize(
    image, None,
    fx=scale_x,
    fy=scale_y,
    interpolation=cv2.INTER_LINEAR
)

plt.imshow(cv2.cvtColor(scaled, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
# AFFINE: X-AXIS SHEAR — general template
image = cv2.imread(IMAGE_PATH)
h, w = image.shape[:2]

kx = 0.5

M = np.float32([
    [1, kx, 0],
    [0, 1,  0]
])

new_width = int(w + abs(kx) * h)

sheared = cv2.warpAffine(image, M, (new_width, h))

plt.imshow(cv2.cvtColor(sheared, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:
# AFFINE: TRANSLATION
image = cv2.imread(IMAGE_PATH)
h, w = image.shape[:2]

tx, ty = 100, 50

M = np.float32([
    [1, 0, tx],
    [0, 1, ty]
])

translated = cv2.warpAffine(image, M, (w, h))

plt.imshow(cv2.cvtColor(translated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

### Transformation matrices to remember

Scaling:

`[[sx, 0], [0, sy]]`

Rotation:

`[[cosθ, -sinθ], [sinθ, cosθ]]`

X-shear:

`[[1, k], [0, 1]]`

Translation needs the affine 2×3 form:

`[[1, 0, tx], [0, 1, ty]]`

Homogeneous coordinates add the extra `1`, allowing translation to be represented inside a 3×3 matrix.

In [ ]:
# PROJECTIVE / PERSPECTIVE TRANSFORMATION
image = cv2.imread(IMAGE_PATH)
h, w = image.shape[:2]

# Source corners in the original image
src = np.float32([
    [0, 0],
    [w-1, 0],
    [w-1, h-1],
    [0, h-1]
])

# Destination corners — CHANGE THESE for the question
dst = np.float32([
    [100, 50],
    [w-150, 20],
    [w-50, h-80],
    [50, h-20]
])

H = cv2.getPerspectiveTransform(src, dst)
perspective = cv2.warpPerspective(image, H, (w, h))

plt.imshow(cv2.cvtColor(perspective, cv2.COLOR_BGR2RGB))
plt.title("Perspective / Homography")
plt.axis("off")
plt.show()

### `warpAffine` vs `warpPerspective`

- `cv2.warpAffine()` → **2×3** affine matrix.
- `cv2.warpPerspective()` → **3×3** homography/projective matrix.

Affine preserves parallelism. Projective can make parallel lines converge, giving perspective.

# LAB 04 — Feature Extraction + Filtering + Convolution + Edge Detection

Manual topics:
- local vs global features
- histogram-based methods
- HOG
- LBP
- color histogram
- HED
- HIG
- texture energy / contrast histograms
- filtering
- convolution and padding/stride
- edge detection
- Canny

In [ ]:
# HOG — scikit-image
from skimage.feature import hog
from skimage import exposure

gray = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)

features, hog_image = hog(
    gray,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    visualize=True
)

hog_vis = exposure.rescale_intensity(hog_image, in_range=(0, 10))

print("HOG feature vector shape:", features.shape)

plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.imshow(gray, cmap="gray"); plt.title("Original"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(hog_vis, cmap="gray"); plt.title("HOG"); plt.axis("off")
plt.show()

### HOG intuition

HOG = Histogram of Oriented Gradients.

Flow:
1. image → usually grayscale
2. calculate gradients
3. divide into cells
4. make orientation histograms
5. group cells into blocks
6. normalize
7. concatenate → feature vector

HOG is especially associated with shape/edge structure and object detection.

In [ ]:
# LBP — texture feature template
from skimage.feature import local_binary_pattern

gray = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)

P = 8
R = 1

lbp = local_binary_pattern(gray, P, R, method="uniform")

plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.imshow(gray, cmap="gray"); plt.title("Gray"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(lbp, cmap="gray"); plt.title("LBP"); plt.axis("off")
plt.show()

### LBP intuition

Center pixel ko neighboring pixels se compare karo.

- neighbor >= center → 1
- neighbor < center → 0

Ye local binary pattern banata hai → texture describe hoti hai.

In [ ]:
# COLOR HISTOGRAM — BGR channels
image = cv2.imread(IMAGE_PATH)

for i, channel in enumerate(["B", "G", "R"]):
    hist = cv2.calcHist([image], [i], None, [256], [0, 256])
    plt.plot(hist, label=channel)

plt.title("Color Histogram")
plt.xlabel("Intensity")
plt.ylabel("Count")
plt.legend()
plt.show()

In [ ]:
# CONVOLUTION — manual kernel example
gray = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)

kernel = np.array([
    [-1, -1, -1],
    [-1,  8, -1],
    [-1, -1, -1]
], dtype=np.float32)

filtered = cv2.filter2D(gray, -1, kernel)

plt.figure(figsize=(10,4))
plt.subplot(1,2,1); plt.imshow(gray, cmap="gray"); plt.title("Original"); plt.axis("off")
plt.subplot(1,2,2); plt.imshow(filtered, cmap="gray"); plt.title("Convolution / Filter"); plt.axis("off")
plt.show()

### Convolution terminology

- **Kernel/filter:** small matrix applied over the image.
- **Stride:** how many pixels the kernel moves at each step.
- **Padding:** extra border around input.
- **Valid:** no padding; output becomes smaller.
- **Same:** padding chosen to preserve spatial size (under suitable stride).
- **Standard convolution:** normal sliding kernel operation.

In [ ]:
# SOBEL EDGE DETECTION
gray = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)

sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

magnitude = cv2.magnitude(
    sobel_x.astype(np.float32),
    sobel_y.astype(np.float32)
)

plt.figure(figsize=(12,4))
plt.subplot(1,3,1); plt.imshow(sobel_x, cmap="gray"); plt.title("Sobel X"); plt.axis("off")
plt.subplot(1,3,2); plt.imshow(sobel_y, cmap="gray"); plt.title("Sobel Y"); plt.axis("off")
plt.subplot(1,3,3); plt.imshow(magnitude, cmap="gray"); plt.title("Magnitude"); plt.axis("off")
plt.show()

In [ ]:
# CANNY — easiest exam template
gray = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)

blurred = cv2.GaussianBlur(gray, (5,5), 1.4)

edges = cv2.Canny(
    blurred,
    50,   # lower threshold
    150   # upper threshold
)

plt.imshow(edges, cmap="gray")
plt.title("Canny Edges")
plt.axis("off")
plt.show()

### Canny pipeline — memorize this order

1. Gaussian smoothing → noise reduction
2. Gradient calculation → magnitude + direction
3. Non-maximum suppression → thin edges
4. Hysteresis thresholding → strong/weak edge tracking

Common code:

`edges = cv2.Canny(image, low, high)`

In [ ]:
# LAPLACIAN OF GAUSSIAN (LoG)
gray = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)

blurred = cv2.GaussianBlur(gray, (5,5), 1.4)

laplacian = cv2.Laplacian(blurred, cv2.CV_64F)
lap_abs = cv2.convertScaleAbs(laplacian)

plt.imshow(lap_abs, cmap="gray")
plt.title("Laplacian of Gaussian (LoG)")
plt.axis("off")
plt.show()

# LAB 05 — Segmentation + Wavelets + Boundary/Hough/SIFT

The merged manual contains a segmentation section and a later section titled:
**Wavelets, Boundary Detection, Hough Transform & SIFT**.

Segmentation methods listed include:
- global thresholding
- adaptive thresholding
- Otsu
- color-based thresholding
- hysteresis/Canny
- edge detection
- region growing
- watershed
- clustering/K-means
- deep-learning segmentation
- active contours

In [ ]:
# SEGMENTATION: GLOBAL THRESHOLDING
gray = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)

_, mask = cv2.threshold(
    gray, 128, 255, cv2.THRESH_BINARY
)

plt.imshow(mask, cmap="gray")
plt.title("Global Threshold")
plt.axis("off")
plt.show()

In [ ]:
# SEGMENTATION: ADAPTIVE THRESHOLDING
gray = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)

adaptive = cv2.adaptiveThreshold(
    gray,
    255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY,
    11,   # block size
    2     # C
)

plt.imshow(adaptive, cmap="gray")
plt.title("Adaptive Threshold")
plt.axis("off")
plt.show()

In [ ]:
# SEGMENTATION: OTSU
gray = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)

_, otsu = cv2.threshold(
    gray, 0, 255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

plt.imshow(otsu, cmap="gray")
plt.title("Otsu")
plt.axis("off")
plt.show()

In [ ]:
# SEGMENTATION: COLOR / HSV RANGE
image = cv2.imread(IMAGE_PATH)
hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

lower = np.array([30, 50, 50])
upper = np.array([60, 255, 255])

mask = cv2.inRange(hsv, lower, upper)

plt.imshow(mask, cmap="gray")
plt.title("HSV Color Mask")
plt.axis("off")
plt.show()

In [ ]:
# SEGMENTATION: K-MEANS — direct exam template
image = cv2.imread(IMAGE_PATH)

pixel_values = image.reshape((-1, 3))
pixel_values = np.float32(pixel_values)

criteria = (
    cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER,
    100,
    0.2
)

K = 4

_, labels, centers = cv2.kmeans(
    pixel_values,
    K,
    None,
    criteria,
    10,
    cv2.KMEANS_RANDOM_CENTERS
)

centers = np.uint8(centers)
segmented = centers[labels.flatten()]
segmented = segmented.reshape(image.shape)

plt.imshow(cv2.cvtColor(segmented, cv2.COLOR_BGR2RGB))
plt.title(f"K-Means Segmentation, K={K}")
plt.axis("off")
plt.show()

### K-means segmentation flow

1. reshape image → pixels
2. convert to float32
3. choose K
4. initialize centers
5. assign each pixel to nearest center
6. recompute centers
7. repeat until convergence
8. reshape back to image

`K` = number of clusters/segments.

In [ ]:
# WATERSHED — compact practical template
# Useful when objects touch each other.
img = cv2.imread(IMAGE_PATH)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

_, thresh = cv2.threshold(
    gray, 0, 255,
    cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
)

kernel = np.ones((3,3), np.uint8)
opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)

sure_bg = cv2.dilate(opening, kernel, iterations=3)

dist = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
_, sure_fg = cv2.threshold(dist, 0.5*dist.max(), 255, 0)

sure_fg = np.uint8(sure_fg)
unknown = cv2.subtract(sure_bg, sure_fg)

_, markers = cv2.connectedComponents(sure_fg)
markers = markers + 1
markers[unknown == 255] = 0

markers = cv2.watershed(img, markers)

result = img.copy()
result[markers == -1] = [0, 0, 255]

plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
plt.title("Watershed")
plt.axis("off")
plt.show()

## Wavelets

- **CWT:** continuous wavelet functions, scaled/translated continuously.
- **DWT:** discrete wavelet functions; common for image compression, denoising, feature extraction.
- **WPT:** extension of DWT with more flexible subband decomposition.
- **Inverse wavelet:** reconstruct original signal/image from wavelet coefficients.

For coding, PyWavelets (`pywt`) is the easiest practical tool.

In [ ]:
# DWT — PyWavelets template
# !pip install PyWavelets
import pywt

gray = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)

coeffs = pywt.dwt2(gray, "haar")
LL, (LH, HL, HH) = coeffs

print("LL:", LL.shape)
print("LH:", LH.shape)
print("HL:", HL.shape)
print("HH:", HH.shape)

plt.figure(figsize=(10,4))
plt.subplot(1,4,1); plt.imshow(LL, cmap="gray"); plt.title("LL"); plt.axis("off")
plt.subplot(1,4,2); plt.imshow(LH, cmap="gray"); plt.title("LH"); plt.axis("off")
plt.subplot(1,4,3); plt.imshow(HL, cmap="gray"); plt.title("HL"); plt.axis("off")
plt.subplot(1,4,4); plt.imshow(HH, cmap="gray"); plt.title("HH"); plt.axis("off")
plt.show()

In [ ]:
# Inverse DWT — reconstruct
reconstructed = pywt.idwt2(
    (LL, (LH, HL, HH)),
    "haar"
)

reconstructed = np.clip(reconstructed, 0, 255).astype(np.uint8)

plt.imshow(reconstructed, cmap="gray")
plt.title("Inverse DWT Reconstruction")
plt.axis("off")
plt.show()

## Hough Transform

Purpose: detect geometric shapes, especially **lines and circles**.

Basic idea:
image/edge pixels vote in a parameter space; strong peaks correspond to likely shapes.

Line representation often taught as `y = mx + b`.

Circle:

`(x-a)^2 + (y-b)^2 = r^2`

For coding, OpenCV gives ready functions.

In [ ]:
# HOUGH LINE DETECTION
gray = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)

edges = cv2.Canny(gray, 50, 150)

lines = cv2.HoughLines(
    edges,
    1,
    np.pi/180,
    100
)

result = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

if lines is not None:
    for line in lines[:20]:
        rho, theta = line[0]
        a = np.cos(theta)
        b = np.sin(theta)
        x0 = a * rho
        y0 = b * rho

        x1 = int(x0 + 1000*(-b))
        y1 = int(y0 + 1000*(a))
        x2 = int(x0 - 1000*(-b))
        y2 = int(y0 - 1000*(a))

        cv2.line(result, (x1,y1), (x2,y2), (0,0,255), 2)

plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
plt.title("Hough Lines")
plt.axis("off")
plt.show()

In [ ]:
# HOUGH CIRCLES
gray = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)
gray_blur = cv2.medianBlur(gray, 5)

circles = cv2.HoughCircles(
    gray_blur,
    cv2.HOUGH_GRADIENT,
    dp=1,
    minDist=20,
    param1=50,
    param2=30,
    minRadius=5,
    maxRadius=100
)

result = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

if circles is not None:
    circles = np.uint16(np.around(circles))
    for x, y, r in circles[0]:
        cv2.circle(result, (x,y), r, (0,255,0), 2)
        cv2.circle(result, (x,y), 2, (0,0,255), 3)

plt.imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
plt.title("Hough Circles")
plt.axis("off")
plt.show()

## SIFT — must know

SIFT = **Scale-Invariant Feature Transform**

Main conceptual stages:
1. scale-space extrema detection
2. keypoint localization
3. orientation assignment
4. descriptor generation
5. keypoint matching
6. outlier rejection (e.g. RANSAC)
7. homography estimation

OpenCV practical core:

`cv2.SIFT_create()`

`detectAndCompute()`

`drawKeypoints()`

In [ ]:
# SIFT — detect keypoints + descriptors
image = cv2.imread(IMAGE_PATH)
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

sift = cv2.SIFT_create()

keypoints, descriptors = sift.detectAndCompute(gray, None)

print("Number of keypoints:", len(keypoints))
print("Descriptor shape:", None if descriptors is None else descriptors.shape)

vis = cv2.drawKeypoints(
    cv2.cvtColor(image, cv2.COLOR_BGR2RGB),
    keypoints,
    None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

plt.figure(figsize=(10,6))
plt.imshow(vis)
plt.title("SIFT Keypoints")
plt.axis("off")
plt.show()

In [ ]:
# SIFT MATCHING — two-image template
img1 = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)
img2 = cv2.imread(IMAGE2_PATH, cv2.IMREAD_GRAYSCALE)

if img1 is None or img2 is None:
    print("Change IMAGE_PATH / IMAGE2_PATH.")
else:
    sift = cv2.SIFT_create()
    kp1, des1 = sift.detectAndCompute(img1, None)
    kp2, des2 = sift.detectAndCompute(img2, None)

    bf = cv2.BFMatcher()
    matches = bf.knnMatch(des1, des2, k=2)

    # Lowe ratio test
    good = []
    for m, n in matches:
        if m.distance < 0.75 * n.distance:
            good.append(m)

    print("Good matches:", len(good))

    matched = cv2.drawMatches(
        img1, kp1,
        img2, kp2,
        good[:50],
        None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )

    plt.figure(figsize=(14,6))
    plt.imshow(matched, cmap="gray")
    plt.title("SIFT Matches")
    plt.axis("off")
    plt.show()

In [ ]:
# SIFT + HOMOGRAPHY + RANSAC — when the question asks for image alignment
img1 = cv2.imread(IMAGE_PATH, cv2.IMREAD_GRAYSCALE)
img2 = cv2.imread(IMAGE2_PATH, cv2.IMREAD_GRAYSCALE)

sift = cv2.SIFT_create()
kp1, des1 = sift.detectAndCompute(img1, None)
kp2, des2 = sift.detectAndCompute(img2, None)

bf = cv2.BFMatcher()
matches = bf.knnMatch(des1, des2, k=2)

good = [m for m,n in matches if m.distance < 0.75*n.distance]

if len(good) >= 4:
    src_pts = np.float32([kp1[m.queryIdx].pt for m in good]).reshape(-1,1,2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in good]).reshape(-1,1,2)

    H, mask = cv2.findHomography(
        src_pts,
        dst_pts,
        cv2.RANSAC,
        5.0
    )

    print("Homography found:", H is not None)
    if mask is not None:
        print("Inliers:", int(mask.sum()), "/", len(mask))
else:
    print("Need at least 4 good matches.")

# EXAM "WHAT FUNCTION DO I USE?" TABLE

| Task | Main function |
|---|---|
| Read image | `cv2.imread()` |
| BGR→RGB | `cv2.cvtColor(..., cv2.COLOR_BGR2RGB)` |
| BGR→Gray | `cv2.cvtColor(..., cv2.COLOR_BGR2GRAY)` |
| Resize | `cv2.resize()` |
| Gaussian blur | `cv2.GaussianBlur()` |
| Crop | NumPy slicing `img[y1:y2, x1:x2]` |
| Text | `cv2.putText()` |
| Threshold | `cv2.threshold()` |
| Adaptive threshold | `cv2.adaptiveThreshold()` |
| Otsu | `cv2.threshold(... + cv2.THRESH_OTSU)` |
| HSV mask | `cv2.inRange()` |
| Rotate | `cv2.getRotationMatrix2D()` + `cv2.warpAffine()` |
| Translate | `cv2.warpAffine()` |
| Affine transform | `cv2.warpAffine()` |
| Perspective | `cv2.getPerspectiveTransform()` + `cv2.warpPerspective()` |
| Blend/add | `cv2.add()` |
| Histogram equalization | `cv2.equalizeHist()` |
| Sobel | `cv2.Sobel()` |
| Canny | `cv2.Canny()` |
| Laplacian | `cv2.Laplacian()` |
| K-means | `cv2.kmeans()` |
| HOG | `skimage.feature.hog()` |
| LBP | `skimage.feature.local_binary_pattern()` |
| DWT | `pywt.dwt2()` |
| Inverse DWT | `pywt.idwt2()` |
| Hough lines | `cv2.HoughLines()` |
| Hough circles | `cv2.HoughCircles()` |
| SIFT | `cv2.SIFT_create()` |
| SIFT descriptors | `sift.detectAndCompute()` |
| Descriptor matching | `cv2.BFMatcher()` |
| Homography + RANSAC | `cv2.findHomography(..., cv2.RANSAC)` |

# EXAM DECISION TREE

If question says...

**"make image grayscale"**
→ `cvtColor`

**"resize / scale"**
→ `resize`

**"smooth / remove noise"**
→ `GaussianBlur`

**"crop / left half / region"**
→ NumPy slicing

**"write text"**
→ `putText`

**"convert to binary"**
→ `threshold`

**"different lighting"**
→ adaptive threshold

**"automatically choose threshold"**
→ Otsu

**"select a color"**
→ HSV + `inRange`

**"rotate / translate / shear"**
→ affine + `warpAffine`

**"perspective / four corners / document flattening"**
→ homography + `warpPerspective`

**"improve global contrast"**
→ histogram equalization

**"edges"**
→ Canny / Sobel / Laplacian

**"segments based on K groups"**
→ K-means

**"texture"**
→ LBP

**"shape / gradient descriptor"**
→ HOG

**"lines"**
→ HoughLines

**"circles"**
→ HoughCircles

**"scale/rotation invariant local features"**
→ SIFT

**"image matching + geometric consistency"**
→ SIFT + BFMatcher + RANSAC + Homography

**"log / gamma / piecewise"**
→ point transformation

# THEORY ONE-PAGER — Roman Urdu + English

### Geometric vs Photometric
**Geometric:** pixel ki location change hoti hai. Example rotation, scaling, translation, shear, perspective.

**Photometric:** pixel ki value/appearance change hoti hai, location same rehti hai. Example brightness, contrast, gamma, log, histogram.

### Affine
Affine transformations preserve **straight lines and parallelism**. Translation, rotation, scaling and shear are common affine operations.

### Projective / Homography
Projective transformation perspective ko model karti hai. Parallel lines image mein converge kar sakti hain. OpenCV mein 3×3 homography + `warpPerspective`.

### HOG
Image ko cells mein divide karke gradient orientations ke histograms banata hai, then blocks normalize karke feature vector banata hai.

### LBP
Center pixel ko neighbors se compare karke local binary pattern banata hai. Texture ke liye useful.

### Canny
Blur → gradient → non-maximum suppression → hysteresis thresholding.

### Segmentation
Image ko meaningful regions/segments mein partition karna.

### K-means
Pixels/features ko K clusters mein group karta hai; nearest centroid assignment + centroid update repeat hota hai.

### Hough
Edge pixels ko parameter space mein vote karwa kar lines/circles detect karta hai.

### SIFT
Multiple scales par stable keypoints find karta hai, orientation assign karta hai, descriptors banata hai, phir matching/RANSAC/homography se images relate ki ja sakti hain.

### DWT
Image ko frequency/subband components mein decompose karta hai. Haar DWT ka common simple example: LL, LH, HL, HH.

# LAST-MINUTE COMMON MISTAKES

1. `image.shape` → `(height, width, channels)`.
2. NumPy image indexing → `image[y, x]`.
3. OpenCV color read → BGR.
4. Matplotlib color display → convert BGR → RGB.
5. `GaussianBlur` kernel usually odd, e.g. `(5,5)`.
6. `cv2.threshold` returns **two values**: `(ret, output)`.
7. `warpAffine` uses 2×3 matrix.
8. `warpPerspective` uses 3×3 matrix.
9. Hough lines usually need an edge image first.
10. SIFT descriptors are numerical feature vectors; keypoints are locations/orientations.
11. K-means needs float32 pixel data.
12. For SIFT matching, BFMatcher + ratio test is a common pattern.
13. RANSAC is for rejecting geometrically inconsistent matches/outliers.
14. Gamma `< 1` → brighter; gamma `> 1` → darker.
15. Histogram = distribution of intensity values; histogram equalization = contrast enhancement.